In [1]:
import pandas as pd

df = pd.read_csv("merged_cleaned_text_PaddleOCR2.csv")


In [2]:
import re

def clean_text(text):
    text = re.sub(r'\[|\]', '', str(text))  # remove square brackets
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    return text

df['text'] = df['text'].apply(clean_text)


In [4]:
import nltk
import tiktoken 
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
tokenizer = tiktoken.get_encoding("cl100k_base")  # Used by OpenAI models

def hybrid_chunking(text, max_tokens=500):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    for sent in sentences:
        temp_chunk = current_chunk + " " + sent if current_chunk else sent
        if len(tokenizer.encode(temp_chunk)) <= max_tokens:
            current_chunk = temp_chunk
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sent
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sneha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
all_chunks = []

for idx, row in df.iterrows():
    file_chunks = hybrid_chunking(row['text'])
    for chunk in file_chunks:
        all_chunks.append({
            "filename": row["filename"],
            "chunk": chunk
        })

chunked_df = pd.DataFrame(all_chunks)


In [8]:
# Save to CSV in the current working directory
chunked_df.to_csv(r"C:\Users\sneha\Downloads\parquet_file\hybrid_chunked_text.csv", index=False)

print("File saved as 'hybrid_chunked_text.csv'.")


File saved as 'hybrid_chunked_text.csv'.


In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import numpy as np

# Load the CSV
chunked_df = pd.read_csv(r"C:\Users\sneha\Downloads\parquet_file\hybrid_chunked_text.csv")

# Initialize sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode text chunks
embeddings = model.encode(chunked_df["chunk"].tolist())

# ---------------------- COSINE SIMILARITY ----------------------

# Compute full cosine similarity matrix
cos_sim_matrix = cosine_similarity(embeddings)

# Mask out diagonal (self-similarity = 1) to get average similarity
mask = ~np.eye(cos_sim_matrix.shape[0], dtype=bool)
avg_cos_sim = cos_sim_matrix[mask].mean()

# ---------------------- BLEU SCORE ----------------------

reference = chunked_df["chunk"][0]
candidate = chunked_df["chunk"][1]

# Use smoothing to avoid zero scores
smoothing = SmoothingFunction().method4
bleu = sentence_bleu([reference.split()], candidate.split(), smoothing_function=smoothing)

# ---------------------- ROUGE SCORE ----------------------

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
rouge_scores = scorer.score(reference, candidate)

# ---------------------- OUTPUT ----------------------

print("🔹 Average Cosine Similarity:", round(avg_cos_sim, 4))
print("🔹 BLEU Score:", round(bleu, 4))
print("🔹 ROUGE-1 Score:", round(rouge_scores['rouge1'].fmeasure, 4))
print("🔹 ROUGE-L Score:", round(rouge_scores['rougeL'].fmeasure, 4))


🔹 Average Cosine Similarity: 0.0676
🔹 BLEU Score: 0.0136
🔹 ROUGE-1 Score: 0.2214
🔹 ROUGE-L Score: 0.0871
